In [1]:
import os
os.environ["DATASETS_DISABLE_TORCHVISION"] = "1"  # обход ImportError с VideoReader
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"  # убираем лишние предупреждения

In [2]:
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

# Фиксация воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device: {device}")
print(f" PyTorch: {torch.__version__}")

 Device: cpu
 PyTorch: 2.11.0+cpu


In [3]:
ds = load_dataset("emotion")  

print(f"✓ Train: {len(ds['train'])}, Val: {len(ds['validation'])}, Test: {len(ds['test'])}")
print(f"✓ Classes: {ds['train'].features['label'].names}")

# Примеры
for i in range(3):
    print(f"[{ds['train']['label'][i]}] {ds['train']['text'][i][:80]}...")

✓ Train: 16000, Val: 2000, Test: 2000
✓ Classes: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
[0] i didnt feel humiliated...
[0] i can go from feeling so hopeless to so damned hopeful just from being around so...
[3] im grabbing a minute to post i feel greedy wrong...


In [4]:
MODEL_NAME = "cointegrated/rubert-tiny2"  

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f" Tokenizer: {tokenizer.__class__.__name__}")

# Пример токенизации
sample_texts = ds["train"]["text"][:3]
for txt in sample_texts:
    enc = tokenizer(txt, truncation=True, max_length=64)
    print(f"\nText: {txt[:50]}...")
    print(f"  Tokens: {tokenizer.convert_ids_to_tokens(enc['input_ids'])[:10]}")
    print(f"  Input IDs: {enc['input_ids'][:10]}")
    print(f"  Attention mask: {enc['attention_mask'][:10]}")
    print(f"  Special tokens: [CLS]={tokenizer.cls_token}, [SEP]={tokenizer.sep_token}")

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

 Tokenizer: BertTokenizerFast

Text: i didnt feel humiliated...
  Tokens: ['[CLS]', 'i', 'didn', '##t', 'feel', 'hu', '##mil', '##iated', '[SEP]']
  Input IDs: [2, 76, 11055, 549, 12235, 8338, 17141, 25292, 3]
  Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1]
  Special tokens: [CLS]=[CLS], [SEP]=[SEP]

Text: i can go from feeling so hopeless to so damned hop...
  Tokens: ['[CLS]', 'i', 'can', 'go', 'from', 'feeling', 'so', 'hope', '##less', 'to']
  Input IDs: [2, 76, 1147, 1695, 610, 18804, 773, 15939, 3425, 540]
  Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  Special tokens: [CLS]=[CLS], [SEP]=[SEP]

Text: im grabbing a minute to post i feel greedy wrong...
  Tokens: ['[CLS]', 'im', 'gra', '##bbi', '##ng', 'a', 'minute', 'to', 'post', 'i']
  Input IDs: [2, 631, 19420, 12169, 770, 68, 6494, 540, 1729, 76]
  Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  Special tokens: [CLS]=[CLS], [SEP]=[SEP]


In [5]:
from transformers import pipeline

demo_pipe = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1
)

test_samples = [
    "I love this product, it works perfectly!",
    "This is terrible, worst experience ever.",
    "It's okay, nothing special."
]

print(" Инференс готовой модели (демо):")
for txt in test_samples:
    res = demo_pipe(txt)[0]
    print(f"  '{txt[:40]}...' → {res['label']} ({res['score']:.3f})")

print("\n Вывод: готовая модель работает в своём пространстве меток.",
      "Для нашей задачи нужен fine-tuning.")

Device set to use cpu


 Инференс готовой модели (демо):
  'I love this product, it works perfectly!...' → POSITIVE (1.000)
  'This is terrible, worst experience ever....' → NEGATIVE (1.000)
  'It's okay, nothing special....' → NEGATIVE (0.819)

 Вывод: готовая модель работает в своём пространстве меток. Для нашей задачи нужен fine-tuning.


In [6]:
# Токенизация всего датасета
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding=False)

tokenized_ds = ds.map(tokenize_fn, batched=True)

# Удаляем текстовую колонку — DataCollator не умеет работать со строками
tokenized_ds = tokenized_ds.remove_columns(["text"])

# Data collator для динамического padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f" Tokenized datasets ready")
print(f"  Train: {len(tokenized_ds['train'])} examples")

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

 Tokenized datasets ready
  Train: 16000 examples


In [7]:
num_labels = len(ds["train"].features["label"].names)
id2label = {i: label for i, label in enumerate(ds["train"].features["label"].names)}
label2id = {label: i for i, label in enumerate(ds["train"].features["label"].names)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # новая классификационная голова
)

# Совместимость: evaluation_strategy → eval_strategy в новых версиях
common_args = dict(
    output_dir="./hw13_checkpoint",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

try:
    training_args = TrainingArguments(
        eval_strategy="epoch",  # новое имя
        save_strategy="epoch",
        **common_args
    )
except TypeError:
    training_args = TrainingArguments(
        evaluation_strategy="epoch",  # старое имя
        save_strategy="epoch",
        **common_args
    )

print(f" Model: {model.__class__.__name__}, Labels: {num_labels}")

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Model: BertForSequenceClassification, Labels: 6


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# Совместимость: tokenizer → processing_class
try:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        processing_class=tokenizer,  # новое имя
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
except TypeError:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        tokenizer=tokenizer,  # старое имя
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

print(" Trainer готов")

 Trainer готов


In [9]:
print(" Запуск fine-tuning...")
train_result = trainer.train()
print(f" Обучение завершено")

C:\Users\Степан\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


 Запуск fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.080300,0.903333,0.704500,0.514595
2,0.669100,0.584591,0.832500,0.767600
3,0.516000,0.514456,0.849000,0.796308


C:\Users\Степан\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\Степан\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


 Обучение завершено


In [10]:
import os
os.makedirs("artifacts", exist_ok=True)

# Оценка на test
test_results = trainer.evaluate(tokenized_ds["test"])
print(f"\n Test Accuracy: {test_results['eval_accuracy']:.4f}")
print(f" Test F1 Macro: {test_results['eval_f1_macro']:.4f}")

# Предсказания для артефактов
test_output = trainer.predict(tokenized_ds["test"])
preds = np.argmax(test_output.predictions, axis=1)
probs = np.max(torch.softmax(torch.tensor(test_output.predictions), dim=-1).numpy(), axis=1)

# sample_predictions.csv
df_pred = pd.DataFrame({
    "text": ds["test"]["text"][:100],
    "true_label": [ds["test"].features["label"].names[l] for l in ds["test"]["label"][:100]],
    "pred_label": [ds["test"].features["label"].names[p] for p in preds[:100]],
    "confidence": np.round(probs[:100], 4),
})
df_pred.to_csv("artifacts/sample_predictions.csv", index=False)
print(" artifacts/sample_predictions.csv")

# confusion_matrix.png
cm = confusion_matrix(ds["test"]["label"], preds)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=ds["train"].features["label"].names,
            yticklabels=ds["train"].features["label"].names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix (Test)")
plt.tight_layout()
plt.savefig("artifacts/confusion_matrix.png", dpi=150)
plt.close()
print(" artifacts/confusion_matrix.png")

C:\Users\Степан\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



 Test Accuracy: 0.8555
 Test F1 Macro: 0.7806


C:\Users\Степан\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


 artifacts/sample_predictions.csv
 artifacts/confusion_matrix.png
